
# 15_CW_extractor.ipynb  
### Self-contained Heavy-Mode Wilson Hessian Constant Extractor

This notebook is fully self-contained.  
You don't need to import any other notebook or module.

It:

- Defines the SU(3) core (generators, exp map, link builder, Wilson action)
- Builds the Wilson-only flat action (no Haar term)
- Implements Hessian–vector product via JVP
- Implements Lanczos minimal eigenvalue extraction
- Performs heavy-mode adversarial search over directions to estimate the curvature constant

Goal:

\[
C_W = \lim_{\theta \to 0} \max_{\|v\|=1}
\left( - \frac{\lambda_{\min}(H_W(\theta v))}{\|\theta v\|^2} \right)
\]

for L = 2, 3, 4.


In [2]:

import jax
import jax.numpy as jnp
from jax import lax
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

jax.config.update("jax_enable_x64", False)


## SU(3) core definitions

In [3]:

def su3_generators():
    lam = []
    lam.append(jnp.array([[0,1,0],[1,0,0],[0,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,-1j,0],[1j,0,0],[0,0,0]], jnp.complex64))
    lam.append(jnp.array([[1,0,0],[0,-1,0],[0,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,1],[0,0,0],[1,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,-1j],[0,0,0],[1j,0,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,0],[0,0,1],[0,1,0]], jnp.complex64))
    lam.append(jnp.array([[0,0,0],[0,0,-1j],[0,1j,0]], jnp.complex64))
    lam.append(jnp.array([[1,0,0],[0,1,0],[0,0,-2]], jnp.complex64)/jnp.sqrt(3.0))
    lam = jnp.stack(lam, axis=0)
    return 1j * lam / 2.0

T_SU3 = su3_generators()

def su3_alg_from_vec(a):
    return jnp.einsum("...a,aij->...ij", a, T_SU3)

@jax.checkpoint
def su3_exp_pade22(A):
    I = jnp.eye(3, dtype=jnp.complex64)
    A2 = A @ A
    Num = I + 0.5 * A + (1.0/12.0) * A2
    Den = I - 0.5 * A + (1.0/12.0) * A2
    return jnp.linalg.solve(Den, Num)

def build_links_factory(L):
    @jax.checkpoint
    def build_links(theta_flat):
        flat = theta_flat.reshape(-1, 8)
        A = jax.vmap(su3_alg_from_vec)(flat)
        U = jax.vmap(su3_exp_pade22)(A)
        return U.reshape(L, L, L, L, 4, 3, 3)
    return build_links

def compute_plaquette_sum(U, beta):
    S = 0.0
    for mu in range(4):
        for nu in range(mu+1, 4):
            U_mu = U[..., mu, :, :]
            U_nu_shift = jnp.roll(U[..., nu, :, :], -1, axis=mu)
            U_mu_dag_shift = jnp.swapaxes(
                jnp.conjugate(jnp.roll(U[..., mu, :, :], -1, axis=nu)),
                -1, -2
            )
            U_nu_dag = jnp.swapaxes(jnp.conjugate(U[..., nu, :, :]), -1, -2)
            P = U_mu @ U_nu_shift @ U_mu_dag_shift @ U_nu_dag
            trP = jnp.real(jnp.einsum("...ii->...", P))
            S += jnp.sum(1.0 - trP/3.0)
    return beta * S

def make_flat_action_wilson_only(L, beta):
    build_links_L = build_links_factory(L)
    def unflatten(x):
        return x.reshape((L, L, L, L, 4, 8))
    @jax.jit
    def flat_action(theta_flat):
        U = build_links_L(theta_flat)
        return compute_plaquette_sum(U, beta)
    return flat_action

def hvp(flat_action, theta, v):
    g = jax.grad(flat_action)
    _, hv = jax.jvp(g, (theta,), (v,))
    return hv

def lanczos_min(flat_action, theta, k=30, seed=0):
    key = jax.random.PRNGKey(seed)
    n = theta.shape[0]
    v0 = jax.random.normal(key, (n,))
    v0 /= jnp.linalg.norm(v0)
    def step(carry, _):
        v_prev, v_cur, beta_prev = carry
        w = hvp(flat_action, theta, v_cur)
        w -= beta_prev * v_prev
        alpha = jnp.dot(w, v_cur)
        w -= alpha * v_cur
        beta = jnp.linalg.norm(w)
        v_next = w / (beta + 1e-9)
        return (v_cur, v_next, beta), (alpha, beta)
    (_, _, _), (a, b) = lax.scan(
        step,
        init=(jnp.zeros_like(v0), v0, 0.0),
        xs=None,
        length=k
    )
    a = jnp.array(a)
    b = jnp.array(b[:-1])
    T = jnp.diag(a) + jnp.diag(b, 1) + jnp.diag(b, -1)
    return float(jnp.linalg.eigvalsh(T)[0])


## Heavy-mode C_W extraction logic

In [4]:

BATCH = 64
THETAS = [0.005, 0.01, 0.02, 0.03, 0.05]
LATTICES = [2, 3, 4]
LANCZOS_K = 30
HC_OUTER = 10    # reduced for sanity
HOTRG_BLOCKS = [16, 32, 64, 128]
SEED = 123

def unit_sphere_batch(key, batch, n):
    keys = jax.random.split(key, batch)
    V = jax.vmap(lambda k: jax.random.normal(k, (n,)))(keys)
    V = V / jnp.linalg.norm(V, axis=1, keepdims=True)
    return V

def hotrg_perturbation(key, n, block):
    if block >= n:
        block = max(1, n//4)
    idx = jax.random.randint(key, (), 0, n-block)
    k1, k2 = jax.random.split(key)
    noise = jax.random.normal(k2, (block,))
    noise = noise / jnp.linalg.norm(noise)
    v = jnp.zeros((n,))
    return v.at[idx:idx+block].set(noise)

def CW_value(flat, theta_amp, v, seed):
    th = theta_amp * v
    lam = lanczos_min(flat, th, k=LANCZOS_K, seed=seed)
    return -lam / float(theta_amp**2)

def optimize_direction(flat, theta_amp, n, key):
    key_init, key = jax.random.split(key)
    v_best = jax.random.normal(key_init, (n,))
    v_best = v_best / jnp.linalg.norm(v_best)
    C_best = CW_value(flat, theta_amp, v_best, 0)

    for outer in range(HC_OUTER):
        key, sub = jax.random.split(key)
        V = unit_sphere_batch(sub, BATCH, n)
        for blk in HOTRG_BLOCKS:
            key, sub = jax.random.split(key)
            pert = hotrg_perturbation(sub, n, blk)
            cand = v_best + 0.1 * pert
            cand = cand / jnp.linalg.norm(cand)
            V = jnp.vstack([V, cand])
        scores = []
        for i in range(V.shape[0]):
            key, sub = jax.random.split(key)
            C = CW_value(flat, theta_amp, V[i], seed=i)
            scores.append(C)
        scores = np.array(scores)
        idx = np.argmax(scores)
        if scores[idx] > C_best:
            C_best = scores[idx]
            v_best = V[idx]
    return v_best, C_best


## Run heavy-mode scan over L and θ

In [ ]:

results = []

for L in LATTICES:
    print(f"=== L = {L} ===")
    n = (L**4)*4*8
    flat = make_flat_action_wilson_only(L, beta=1.0)  # Wilson only
    for theta_amp in THETAS:
        key = jax.random.PRNGKey(SEED + int(1000*theta_amp) + L)
        v_best, C_best = optimize_direction(flat, theta_amp, n, key)
        results.append((L, theta_amp, C_best))
        print(f"L={L} θ={theta_amp}  C_W={C_best:+.6f}")

df = pd.DataFrame(results, columns=["L","theta","C_W"])
out_csv = "/mnt/data/CW_raw_results.csv"
df.to_csv(out_csv, index=False)
print("Saved", out_csv)


=== L = 2 ===


/usr/local/lib/python3.12/dist-packages/jax/_src/lax/lax.py:5473: ComplexWarning: Casting complex values to real discards the imaginary part
  x_bar = _convert_element_type(x_bar, x.aval.dtype, x.aval.weak_type)


## Fit C_W(θ) → C₀ as θ→0

In [1]:

fits = []
for L in LATTICES:
    sub = df[df["L"]==L]
    thetas = np.array(sub["theta"])
    Cvals  = np.array(sub["C_W"])
    X = np.vstack([np.ones_like(thetas), thetas**2]).T
    C0, slope = np.linalg.lstsq(X, Cvals, rcond=None)[0]
    fits.append((L, C0))
    print(f"L={L}: C_W(θ→0) ≈ {C0:.6f}")

fit_df = pd.DataFrame(fits, columns=["L","C0"])
fit_df.to_csv("/mnt/data/CW_theta_fit.csv", index=False)

summary_path = "/mnt/data/CW_constant_summary.txt"
with open(summary_path, "w") as f:
    for L, C0 in fits:
        f.write(f"L={L}: C_W ≈ {C0:.6f}\n")
print("Saved", summary_path)


NameError: name 'LATTICES' is not defined

## Diagnostic plots per L

In [ ]:

for L in LATTICES:
    sub = df[df["L"]==L]
    thetas = np.array(sub["theta"])
    Cvals  = np.array(sub["C_W"])
    C0 = fit_df[fit_df["L"]==L]["C0"].iloc[0]
    plt.figure(figsize=(6,4))
    plt.scatter(thetas**2, Cvals, label="samples")
    xs = np.linspace(0, max(thetas)**2, 100)
    plt.plot(xs, C0 + 0*xs, label=f"C_W ≈ {C0:.2f}")
    plt.xlabel("theta^2")
    plt.ylabel("C_W")
    plt.title(f"L={L} Heavy-Mode C_W Extraction")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()
